# Exp 11: K-Means / Hierarchical Clustering on sales_data_sample.csv (Elbow method)
**Dataset:** `sales_data_sample.csv` from https://www.kaggle.com/datasets/kyanyoga/sample-sales-data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage

In [ ]:
df = pd.read_csv("sales_data_sample.csv", encoding="unicode_escape")   # file is not UTF-8
print(df.shape)
print(df.head())
print(df.isnull().sum())

In [ ]:
# Use numeric features for clustering
features = ["QUANTITYORDERED", "PRICEEACH", "ORDERLINENUMBER", "SALES", "MSRP"]
features = [c for c in features if c in df.columns]
X = df[features].dropna()

X_scaled = StandardScaler().fit_transform(X)
print(X_scaled[:5])

## Elbow method to find number of clusters

In [ ]:
wcss = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

plt.plot(range(1, 11), wcss, marker="o")
plt.title("Elbow Method"); plt.xlabel("Number of clusters (k)"); plt.ylabel("WCSS")
plt.grid(True); plt.show()

## K-Means with the chosen k (look at the elbow in the plot above; k = 3 is typical for this dataset)

In [ ]:
k = 3
kmeans = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
labels = kmeans.fit_predict(X_scaled)
X_out = X.copy()
X_out["Cluster"] = labels
print(X_out["Cluster"].value_counts())
print(X_out.groupby("Cluster").mean())

pca = PCA(n_components=2)
pts = pca.fit_transform(X_scaled)
plt.scatter(pts[:, 0], pts[:, 1], c=labels, cmap="viridis", s=10)
centers = pca.transform(kmeans.cluster_centers_)
plt.scatter(centers[:, 0], centers[:, 1], c="red", marker="X", s=200, label="Centroids")
plt.title("K-Means clusters (PCA view)"); plt.legend(); plt.show()

## Hierarchical clustering (dendrogram on a sample)

In [ ]:
sample = X_scaled[:200]
Z = linkage(sample, method="ward")
plt.figure(figsize=(10, 5))
dendrogram(Z)
plt.title("Dendrogram"); plt.xlabel("Samples"); plt.ylabel("Euclidean distance")
plt.show()

hc = AgglomerativeClustering(n_clusters=3, linkage="ward")
hc_labels = hc.fit_predict(sample)
print(pd.Series(hc_labels).value_counts())

**Conclusion:** The elbow method was used to choose k, and K-Means / hierarchical clustering grouped the sales data into clusters.